# ImmuVis trained on Jackson, tested on Danenberg (cross-cohort virtual staining)

**What this run does:**
- Trains the **published ImmuVis code** from scratch on the Jackson train patients and tests it on the Danenberg test patients.
  - Code: Uchal et al. 2026, `anonymous.4open.science/r/ImmuVis-9173`, with `github.com/dav3794/multiplex-image-model` as a fallback.
  - Architecture: the default `train_masked_config.yaml`, i.e. **ImmuVis-Conv** (ConvNeXt marker-agnostic + pan-marker encoder, hyperkernels, heteroscedastic decoder).
- The public ImmuVis weights are not used. They were pretrained on IMC17M, which contains both cohorts.

**Channels:** the 14 channels both cohorts share (H3, DNA1, DNA2 + the 11 shared markers), each as its own marker token.
- At test time each marker is predicted from the other 13, the same inputs as every other model in the figure.

**Preprocessing (ImmuVis pipeline):**
- arcsinh(x/5) → Butterworth low-pass (cutoff 0.2) → clip at 2.3 → /2.3.
- 2.3 is the published Jackson-Basel clip limit. It is used for Danenberg too, so no Danenberg statistics are used (level A).

**Training:**
- **Loss and masking**, as published: random 128×128 crops with rotation and flips; channel-subset sampling (≥ 75%); up to 50% of channels fully masked; 60% spatial masking with 8-px patches; β-NLL loss (β = 0.5); AdamW (lr 5e-4 → 1e-5, cosine, 10% warm-up), weight decay 1e-4.
- **Changes from the published setup:**
  - fp16 autocast, because a T4 has no bf16;
  - a time budget sets the number of steps;
  - the checkpoint is chosen by leave-one-marker-out pixel r on Jackson val patients.
- **Patients:** the same Jackson train/val patients as the Jackson-trained per-cell models (replicate 0 of `train-test-grid`).

**Inference:** 128-px tiles with 32-px context, stitched. Scored exactly like `cross-cohort-metrics`.

**Two kernels:** `immuvis-jackson-train` trains and saves the checkpoint; `immuvis-jackson` loads it and scores Danenberg. (The first single-kernel run was killed by Kaggle after 5.5 h with no log, losing the checkpoint.)

In [ ]:
import os, glob, json, time, sys, subprocess, warnings, traceback, re, pickle, gc, random
import numpy as np, pandas as pd
import pyarrow.parquet as pq
import scipy.sparse as sp
from scipy.spatial import cKDTree
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LinearRegression
from joblib import Parallel, delayed
try:
    import lightgbm as lgb
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lightgbm'], check=True)
    import lightgbm as lgb
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

IN = os.environ.get("IN_ROOT", "/kaggle/input")
OUT = os.environ.get("OUT_DIR", "/kaggle/working")
SMOKE = os.environ.get("SMOKE") == "1"
SEED = 20261013
N_BOOT = 100 if SMOKE else 1000
N_BOOT_F1 = 50 if SMOKE else 200
N_SHIFT = 5
MIN_CELLS = 200
COF = 5.0
RING_R = 30.0
N_FIT = 15_000 if SMOKE else 100_000          # cells per cross-fit half (R5) and per classifier
N_FIT_FINAL = 30_000 if SMOKE else 300_000    # Jackson cells for the baselines
LGB = dict(n_estimators=40 if SMOKE else 250, learning_rate=0.06, num_leaves=31, min_child_samples=50,
           subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, n_jobs=os.cpu_count(), verbose=-1)
ARMS = ["residual", "gamma", "immuvis"]
NUC = ["histone_h3", "dna1", "dna2"]
# Image channel orders (as in AbundanceVsStructure). Danenberg index 12 'c_erb_b_2_her2' is clone 3B5.
D_CHAN = ['histone_h3', 'sma', 'cytokeratin_5', 'cd38', 'hla_dr', 'cytokeratin_8_18', 'cd15', 'fsp1', 'cd163',
          'cd278_icos', 'cd134', 'cd68', 'c_erb_b_2_her2', 'cd3', 'podoplanin', 'cd11c', 'cd279_pd_1', 'gitr_tnfrsf18',
          'cd16', 'c_erb_b_2_her2_d8f12', 'cd45ra', 'beta_2_microglobulin', 'cd45', 'foxp3', 'cd20',
          'estrogen_receptor_alpha', 'cd8a', 'cd57', 'ki_67', 'cd140b_pdgf_receptor_beta', 'caveolin_1', 'cd4',
          'cd31_v_wf', 'cxcl12_sdf_1', 'hla_abc', 'pan_cytokeratin', 'cleaved_caspase3', 'dna1', 'dna2']
J_CHAN = ['histone_h3', 'histone_h3_trimethylate', 'cytokeratin_5', 'fibronectin', 'cytokeratin_19', 'cytokeratin_8_18',
          'twist', 'cd68', 'keratin_14_krt14', 'sma', 'vimentin', 'c_myc', 'c_erb_b_2_her2', 'cd3', 'histone_h3_phospho',
          'slug', 'estrogen_receptor_alpha', 'progesterone_receptor_a_b', 'p53', 'cd44', 'cd45', 'gata3', 'cd20',
          'carbonic_anhydrase_ix', 'e_cadherin_p_cadherin', 'ki_67', 'egfr', 's6', 'v_wf', 'm_tor', 'cytokeratin_7',
          'pan_cytokeratin', 'cleaved_parp', 'dna1', 'dna2']
D_TABLE = {"c_erb_b_2_her2": "c_erb_b_2_her2_3b5"}       # image name -> Danenberg cell-table column
MARKERS = sorted(m for m in set(D_CHAN) & set(J_CHAN) if m not in NUC)    # = task_names in the training code
assert len(MARKERS) == 11, MARKERS
STROMAL_KEYS = ("fibro", "endothel", "stroma", "pdpn", "myofib", "vessel", "vwf")
T0 = time.time()

def tick(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} min] {msg}", flush=True)

def g(pattern):
    return sorted(glob.glob(f"{IN}/**/{pattern}", recursive=True))

def near(name, anchor):
    hits = g(name)
    assert hits, f"missing input {name}"
    return max(hits, key=lambda c: len(os.path.commonpath([c, anchor])))

def guard(name):
    """Run an analysis block; on failure print the traceback and carry on, so the kernel ends COMPLETE."""
    def deco(fn):
        try:
            return fn()
        except Exception:
            print(f"!!! block {name} failed:\n{traceback.format_exc()}", flush=True)
            FAILED.append(name)
            return None
    return deco
FAILED = []

## Cell tables (Danenberg cells to analyse; Jackson cells for the baselines)

In [ ]:
P = {}
for p in g("cells.parquet"):
    key = "danenberg" if "metabric_id" in pq.read_schema(p).names else "jackson"
    P[key] = dict(cells=p, contacts=near("cell_neighbours.parquet", p), report=near("prep_report.json", p))
P["danenberg"].update(clinical=near("clinical.parquet", P["danenberg"]["cells"]),
                      split=near("danenberg_patient_split.csv", P["danenberg"]["cells"]))
print(json.dumps(P, indent=1))

def load_table(key):
    cells = pd.read_parquet(P[key]["cells"])
    if key == "danenberg":
        cells = cells[(cells.is_tumour == 1) & (cells.is_hot_aggregate == 0)]
        pid = "metabric_id"
    else:
        cells = cells[cells.is_tumour == 1]
        pid = "patient_id"
    n_per = cells.groupby("image_name").size()
    cells = cells[cells.image_name.isin(n_per.index[n_per >= MIN_CELLS])]
    cells = cells.sort_values(["image_name", "object_number"]).reset_index(drop=True)
    img_code, img_names = pd.factorize(cells.image_name, sort=True)
    starts = np.r_[0, np.flatnonzero(np.diff(img_code)) + 1, len(cells)]
    loc = pd.Series(np.arange(len(cells)), index=pd.MultiIndex.from_arrays(
        [cells.image_number.values, cells.object_number.values]))
    con = pd.read_parquet(P[key]["contacts"])
    ca = loc.reindex(pd.MultiIndex.from_arrays([con.ImageNumber.values, con["from"].values])).values
    cb = loc.reindex(pd.MultiIndex.from_arrays([con.ImageNumber.values, con["to"].values])).values
    ok = ~(np.isnan(ca) | np.isnan(cb))
    ca, cb = ca[ok].astype(np.int64), cb[ok].astype(np.int64)
    ca, cb = ca[ca != cb], cb[ca != cb]
    if key == "danenberg":
        epi_frac = cells.groupby("cell_phenotype").is_epithelial.mean()
        bmap = {p: "epithelial" if epi_frac[p] > 0.5 else
                "stromal" if any(s in p.lower() for s in STROMAL_KEYS) else "immune" for p in epi_frac.index}
        broad = cells.cell_phenotype.map(bmap).astype(str).values
        epi = cells.is_epithelial.values == 1
    else:
        broad = cells.broad_class.astype(str).values
        epi = broad == "epithelial"
    return dict(key=key, cells=cells, img=img_code, img_names=np.asarray(img_names), starts=starts,
                patient=cells[pid].astype(str).values, xy=cells[["x", "y"]].values.astype(np.float64),
                ca=ca, cb=cb, broad=broad, epi=epi, phen=cells.cell_phenotype.astype(str).values,
                log_area=np.log(np.maximum(cells["area"].values, 1)).astype(np.float32))

DD, DJ = load_table("danenberg"), load_table("jackson")
N = len(DD["cells"])
split = pd.read_csv(P["danenberg"]["split"]).set_index("metabric_id").split
DD["split"] = pd.Series(DD["patient"]).map(split).fillna("none").values
clin = pd.read_parquet(P["danenberg"]["clinical"]).replace({"nan": np.nan})
clin = clin.assign(pid=clin.metabric_id.astype(str)).groupby("pid").first()
print(f"Danenberg: {N:,} cells, {len(DD['img_names'])} images, {len(set(DD['patient']))} patients; "
      f"split cells {pd.Series(DD['split']).value_counts().to_dict()}")
print(f"Jackson: {len(DJ['cells']):,} cells, {len(DJ['img_names'])} images")
pats, pcode = np.unique(DD["patient"], return_inverse=True)
W = np.random.default_rng(SEED).multinomial(len(pats), np.full(len(pats), 1 / len(pats)), size=N_BOOT).astype(np.float64)
tick("tables loaded")

def contact_matrix(D):
    n = len(D["cells"])
    C = sp.coo_matrix((np.ones(len(D["ca"]), np.float32), (D["ca"], D["cb"])), shape=(n, n)).tocsr()
    C = (C + C.T).tocsr()
    C.data[:] = 1.0
    return C

def ring_means(X, D, C):
    """Mean of X over non-touching cells within RING_R of the centroid, and their count."""
    n, p = X.shape
    Mr = np.full((n, p), np.nan, np.float32)
    nr = np.zeros(n, np.float32)
    for a, b in zip(D["starts"][:-1], D["starts"][1:]):
        m = b - a
        pr = cKDTree(D["xy"][a:b]).query_pairs(RING_R, output_type="ndarray")
        R = sp.coo_matrix((np.ones(len(pr), np.float32), (pr[:, 0], pr[:, 1])), shape=(m, m)).tocsr()
        R = (R + R.T).tocsr()
        Ci = C[a:b, a:b].tocsr()
        R = (R - R.multiply(Ci)).tocsr()
        R.eliminate_zeros()
        dr = np.asarray(R.sum(1)).ravel()
        Mr[a:b] = np.where(dr[:, None] > 0, (R @ X[a:b]) / np.maximum(dr, 1)[:, None], np.nan)
        nr[a:b] = dr
    return Mr, nr

## Danenberg test set, truth maps and scoring (shared with `cross-cohort-metrics`)

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "imagecodecs==2024.9.22"], check=True)
import tifffile
import torch
dev = "cuda" if torch.cuda.is_available() else "cpu"
N_GPU = torch.cuda.device_count()

# truth cell means (target space) from kill-test, in cell-table order
KT = pd.read_parquet(g("percell_truth_pred.parquet")[0], columns=["image_name", "object_number"] + [f"truth__{m}" for m in MARKERS])
assert (KT.image_name.values == DD["cells"].image_name.values).all() and \
       (KT.object_number.values == DD["cells"].object_number.values).all(), "kill-test rows do not match the cell table"
TRUTH = {m: KT[f"truth__{m}"].values.astype(np.float32) for m in MARKERS}
del KT

tifs = g("*.tiff")
D_IMG = {os.path.splitext(os.path.basename(p))[0]: p for p in tifs
         if os.path.basename(p).startswith("MB_") and "/masks/" not in p.replace("\\", "/")}
D_MASK = {os.path.splitext(os.path.basename(p))[0]: p for p in tifs
          if os.path.basename(p).startswith("MB_") and "/masks/" in p.replace("\\", "/")}
J_IMG = {os.path.splitext(os.path.basename(p))[0]: p for p in tifs
         if not os.path.basename(p).startswith("MB_") and "/masks/" not in p.replace("\\", "/")
         and "jackson" in p.lower()}
IMGS = list(DD["img_names"])
OBJ = DD["cells"].object_number.values.astype(np.int64)
ST = DD["starts"]
D_TE = np.flatnonzero(DD["split"] == "test")
IMG_TE = DD["img"][D_TE]
TEST_IMGS = np.unique(IMG_TE)
dix = {m: i for i, m in enumerate(D_CHAN)}
jix = {m: i for i, m in enumerate(J_CHAN)}
CACHE = os.path.dirname(g("channel_preproc_residual.pkl")[0])
_, TNS = pickle.load(open(f"{CACHE}/channel_preproc_residual.pkl", "rb"))
PIXK = pd.read_csv(g("pixel_sums_per_image.csv")[0])
PIXK = PIXK[PIXK.arm == "residual"].set_index(["marker", "image"])
print(f"Danenberg test: {len(D_TE):,} cells in {len(TEST_IMGS)} images; Jackson tiffs found: {len(J_IMG)}")

# Jackson split: replicate 0 of train-test-grid (same patients as the Jackson-trained per-cell models)
jp, jcode = np.unique(DJ["patient"], return_inverse=True)
_u = np.random.default_rng(SEED + 11 + 0).random(len(jp))
J_SPLIT = np.where(_u < 0.7, "train", np.where(_u < 0.8, "val", "test"))[jcode]
J_IMG_SPLIT = pd.Series(J_SPLIT, index=DJ["cells"].image_name.values).groupby(level=0).first()
J_TRAIN = sorted(s for s in J_IMG_SPLIT.index[J_IMG_SPLIT == "train"] if s in J_IMG)
J_VAL = sorted(s for s in J_IMG_SPLIT.index[J_IMG_SPLIT == "val"] if s in J_IMG)
print(f"Jackson images: train {len(J_TRAIN)}, val {len(J_VAL)} (tumour images of train / val patients)")

def read_hwc(path, n_chan):
    im = tifffile.imread(path)
    if im.ndim == 3 and im.shape[0] == n_chan and im.shape[-1] != n_chan:
        im = np.moveaxis(im, 0, -1)
    assert im.shape[-1] == n_chan, (path, im.shape)
    return im.astype(np.float32)

def truth_map(im_hwc, m):
    """The image models' target space: clip(arcsinh(x/5) / p99_Jackson, 0, 1) (p01 = 0)."""
    p01, p99 = TNS[jix[m]]
    return np.clip((np.arcsinh(im_hwc[..., dix[m]] / COF) - p01) / (p99 - p01 + 1e-8), 0., 1.)

def cell_index(i):
    """Pixel -> row in the test arrays, for test image i."""
    m = tifffile.imread(D_MASK[IMGS[i]]).astype(np.int64)
    a, b = ST[i], ST[i + 1]
    lut = np.full(max(int(m.max()), int(OBJ[a:b].max())) + 1, -1, np.int64)
    lut[OBJ[a:b]] = np.searchsorted(D_TE, np.arange(a, b))
    return lut[m]

def r_from(n, Sg, Sp, Sgg, Spp, Sgp):
    with np.errstate(invalid="ignore", divide="ignore"):
        return (Sgp - Sg * Sp / n) / np.sqrt((Sgg - Sg ** 2 / n) * (Spp - Sp ** 2 / n))

def r4_point(a, b):
    ok = np.isfinite(a) & np.isfinite(b)
    a, b, im = a[ok].astype(np.float64), b[ok].astype(np.float64), IMG_TE[ok]
    k = int(im.max()) + 1
    cnt = np.maximum(np.bincount(im, minlength=k), 1)
    a = a - (np.bincount(im, a, k) / cnt)[im]
    b = b - (np.bincount(im, b, k) / cnt)[im]
    return float((a * b).sum() / np.sqrt((a * a).sum() * (b * b).sum()))

def score_model(MODEL, predict_image):
    """predict_image(im_hwc) -> {marker: prediction map (H, W)}. Writes pixel sums and per-cell predictions."""
    rows = []
    CELLP = {m: np.full(len(D_TE), np.nan, np.float32) for m in MARKERS}
    for j, i in enumerate(TEST_IMGS):
        s = IMGS[i]
        im = read_hwc(D_IMG[s], len(D_CHAN))
        preds = predict_image(im)
        loc = cell_index(i)
        v = loc >= 0
        for m in MARKERS:
            G = truth_map(im, m).astype(np.float64).ravel()
            Pm = preds[m].astype(np.float64)
            assert Pm.shape == im.shape[:2], (s, m, Pm.shape)
            Pf = Pm.ravel()
            row = dict(model=MODEL, marker=m, image=s, n=G.size, Sg=G.sum(), Sgg=G @ G, Sp=Pf.sum(), Spp=Pf @ Pf, Sgp=G @ Pf)
            if j < 3:   # the truth map must equal kill-test's
                k_ = PIXK.loc[(m, s)]
                assert abs(row["Sg"] - k_.Sg) <= 1e-4 * max(1.0, abs(k_.Sg)) and row["n"] == k_.n, (s, m, row["Sg"], k_.Sg)
            rows.append(row)
            cid = loc[v]
            cnt = np.bincount(cid, minlength=len(D_TE))
            sm = np.bincount(cid, Pm[v], len(D_TE))
            got = cnt > 0
            CELLP[m][got] = (sm[got] / cnt[got]).astype(np.float32)
        if j % 20 == 0:
            tick(f"{MODEL}: scored {j + 1}/{len(TEST_IMGS)} test images")
    S = pd.DataFrame(rows)
    tag = MODEL.lower().replace(" ", "-")
    S.to_csv(f"{OUT}/pixel_sums_test_{tag}.csv", index=False)
    np.savez_compressed(f"{OUT}/percell_pred_{tag}.npz", image=np.asarray(IMGS)[IMG_TE], object_number=OBJ[D_TE],
                        **{f"pred__{MODEL}__{m}": CELLP[m] for m in MARKERS})
    # point summaries (CIs are computed locally with the other models, same bootstrap draws)
    out = []
    for m in MARKERS:
        d = S[S.marker == m]
        ri = r_from(d.n.values, d.Sg.values, d.Sp.values, d.Sgg.values, d.Spp.values, d.Sgp.values)
        out.append(dict(marker=m, pooled=float(r_from(*(d[c].sum() for c in ["n", "Sg", "Sp", "Sgg", "Spp", "Sgp"]))),
                        per_image=float(np.nanmean(ri)), R4=r4_point(TRUTH[m][D_TE], CELLP[m])))
    out = pd.DataFrame(out)
    out.to_csv(f"{OUT}/metrics_point_{tag}.csv", index=False)
    print(out.round(3).to_string(index=False))
    print("mean over 10 markers (ERα excluded):",
          out[out.marker != "estrogen_receptor_alpha"][["pooled", "per_image", "R4"]].mean().round(3).to_dict())
    return S, CELLP

## ImmuVis code and model configuration

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "comet_ml", "ruamel.yaml"], check=True)
import zipfile, urllib.request, copy
from torch.amp import autocast, GradScaler
import torchvision.transforms.functional as TF
from skimage.filters import butterworth

CODE = "/tmp/immuvis_code"
def fetch_code():
    try:
        urllib.request.urlretrieve("https://anonymous.4open.science/api/repo/ImmuVis-9173/zip", "/tmp/immuvis.zip")
        zipfile.ZipFile("/tmp/immuvis.zip").extractall(CODE)
        return "anonymous.4open.science/r/ImmuVis-9173"
    except Exception as e:
        print("anonymous release not available:", e)
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/dav3794/multiplex-image-model.git", CODE], check=True)
        return "github.com/dav3794/multiplex-image-model"
CODE_SOURCE = fetch_code()
root = CODE if os.path.isdir(f"{CODE}/multiplex_model") else os.path.dirname(glob.glob(f"{CODE}/**/multiplex_model", recursive=True)[0])
sys.path.insert(0, root)
from multiplex_model.modules import MultiplexAutoencoder
from multiplex_model.utils import (EncoderConfig, DecoderConfig, apply_channel_masking, apply_spatial_masking,
                                   ClampWithGrad, get_scheduler_with_warmup)
from multiplex_model.losses import beta_nll_loss
print("ImmuVis code from", CODE_SOURCE)
# cuDNN on a T4 ran this model at ~6 s/step and leaked ~50 MB host RAM per step (immuvis-profile);
# with cuDNN off: 0.58 s/step and flat RAM
torch.backends.cudnn.enabled = False

CH = NUC + MARKERS                         # 14 shared channels = 14 marker tokens
IDS = torch.arange(len(CH))
CLIP = 2.3
CROP, BATCH = 128, 16
TRAIN_MIN = float(os.environ.get("TRAIN_MIN", 180))
EVAL_EVERY = 500

def immuvis_pre(im_hwc, names, idx):
    x = np.arcsinh(np.stack([im_hwc[..., idx[c]] for c in names]) / COF)
    x = np.stack([butterworth(c, cutoff_frequency_ratio=0.2, high_pass=False) for c in x])
    return (np.clip(x, 0, CLIP) / CLIP).astype(np.float16)

ENC = EncoderConfig(ma_layers_blocks=[4], ma_embedding_dims=[16], pm_layers_blocks=[4, 4, 4],
                    pm_embedding_dims=[192, 384, 768],
                    hyperkernel=dict(kernel_size=1, padding=0, stride=1, use_bias=True)).model_dump()
DEC = DecoderConfig(decoded_embed_dim=512, num_blocks=1,
                    hyperkernel=dict(kernel_size=1, padding=0, stride=1, use_bias=True)).model_dump()

## Jackson data and training (saves the checkpoint)

In [ ]:
import psutil, shutil, signal
# Training runs in a child process. Two runs of this notebook died with an empty log; here the notebook only
# preprocesses, frees its tables, launches the child and reports its progress and memory every minute, so a crash
# leaves a trace (exit signal, last progress lines, memory).

def sysmem():
    vm = psutil.virtual_memory()
    return f"system RAM used {(vm.total - vm.available) / 2**30:.1f} of {vm.total / 2**30:.1f} GB"

def prep_jackson(s):
    return immuvis_pre(read_hwc(J_IMG[s], len(J_CHAN)), CH, jix)

def save_flat(names, path):
    """Preprocessed images, concatenated into one float16 file (memory-mapped by the child) plus their shapes."""
    arrs = Parallel(n_jobs=4)(delayed(prep_jackson)(s) for s in names)
    shapes = np.array([a.shape[1:] for a in arrs], np.int64)
    flat = np.concatenate([a.reshape(len(CH), -1) for a in arrs], axis=1)
    np.save(f"{path}_flat.npy", flat)
    np.save(f"{path}_shapes.npy", shapes)
    return len(arrs)

N_TR = save_flat(J_TRAIN, "/tmp/jtr")
N_VA = save_flat(J_VAL, "/tmp/jva")
from joblib.externals.loky import get_reusable_executor
get_reusable_executor().shutdown(wait=True)
for _v in ["DD", "DJ", "TRUTH", "KT", "PIXK", "clin"]:
    globals().pop(_v, None)
gc.collect()
tick(f"Jackson preprocessed ({N_TR} train, {N_VA} val images); {sysmem()}")

SCRIPT = r"""
import os, sys, json, time, gc, traceback
cfg = json.loads(sys.argv[1])
import numpy as np, pandas as pd, torch, psutil
import torchvision.transforms.functional as TF
from torch.amp import autocast, GradScaler
sys.path.insert(0, cfg["root"])
from multiplex_model.modules import MultiplexAutoencoder
from multiplex_model.utils import (EncoderConfig, DecoderConfig, apply_channel_masking, apply_spatial_masking,
                                   ClampWithGrad, get_scheduler_with_warmup)
from multiplex_model.losses import beta_nll_loss
torch.backends.cudnn.enabled = False        # see immuvis-profile: cuDNN on a T4 ran ~6 s/step and leaked host RAM
OUT, CH, CLIP, CROP, BATCH = cfg["out"], cfg["channels"], cfg["clip"], cfg["crop"], cfg["batch"]
C = len(CH)
LOGF = open(f"{OUT}/train_progress.log", "a", buffering=1)
T0 = time.time()

def log(msg):
    vm = psutil.virtual_memory()
    LOGF.write(f"[{(time.time() - T0) / 60:6.1f} min] {msg} | RSS {psutil.Process().memory_info().rss / 2**30:.1f} GB, "
               f"system used {(vm.total - vm.available) / 2**30:.1f} GB, GPU0 {torch.cuda.memory_allocated(0) / 2**30:.1f} GB\n")

def load(path):
    flat = np.load(f"{path}_flat.npy", mmap_mode="r")
    shapes = np.load(f"{path}_shapes.npy")
    offs = np.r_[0, np.cumsum(shapes[:, 0] * shapes[:, 1])]
    return [flat[:, offs[i]:offs[i + 1]].reshape(C, *shapes[i]) for i in range(len(shapes))]

JTR, JVA = load("/tmp/jtr"), load("/tmp/jva")
dev = "cuda"
N_GPU = torch.cuda.device_count()
SEED = cfg["seed"]
IDS = torch.arange(C)
_rv = np.random.default_rng(7)
VCROPS = []
for _ in range(96):
    x = JVA[_rv.integers(len(JVA))]
    H, W = x.shape[1:]
    if H < CROP or W < CROP:
        continue
    y0, x0 = _rv.integers(0, H - CROP + 1), _rv.integers(0, W - CROP + 1)
    VCROPS.append(np.asarray(x[:, y0:y0 + CROP, x0:x0 + CROP]))
VCROPS = torch.from_numpy(np.stack(VCROPS).astype(np.float32))
ENC = EncoderConfig(**cfg["enc_args"]).model_dump()
DEC = DecoderConfig(**cfg["dec_args"]).model_dump()
_rng = np.random.default_rng(SEED)
WIN = int(np.ceil(CROP * np.sqrt(2))) + 2

def train_batch():
    out = []
    for _ in range(BATCH):
        x = JTR[_rng.integers(len(JTR))]
        H, W = x.shape[1:]
        if H < WIN or W < WIN:
            x = np.pad(x, ((0, 0), (0, max(0, WIN - H)), (0, max(0, WIN - W))))
            H, W = x.shape[1:]
        y0, x0 = _rng.integers(0, H - WIN + 1), _rng.integers(0, W - WIN + 1)
        out.append(np.asarray(x[:, y0:y0 + WIN, x0:x0 + WIN]))
    t = torch.from_numpy(np.stack(out).astype(np.float32)).to(dev)
    res = []
    for b in range(BATCH):                       # RandomRotation(180, bilinear) -> crop -> RandomHorizontalFlip
        r = TF.center_crop(TF.rotate(t[b], float(_rng.uniform(-180, 180)), interpolation=TF.InterpolationMode.BILINEAR),
                           [CROP, CROP])
        res.append(torch.flip(r, dims=[-1]) if _rng.random() < 0.5 else r)
    return torch.stack(res)

def build():
    torch.manual_seed(SEED)
    net = MultiplexAutoencoder(num_channels=C, encoder_config=ENC, decoder_config=DEC).to(dev)
    return net, (torch.nn.DataParallel(net) if N_GPU > 1 else net)

@torch.no_grad()
def validate(net):
    net.eval()
    rs = []
    for k in range(3, C):                        # leave one marker out; nuclear channels always given
        act = [i for i in range(C) if i != k]
        preds = []
        for b0 in range(0, len(VCROPS), 32):
            xb = VCROPS[b0:b0 + 32].to(dev)
            with autocast(device_type="cuda", dtype=torch.float16):
                lat = net.encode(xb[:, act], IDS[act].to(dev).expand(len(xb), -1))["output"]
                o = net.decode(lat, IDS[[k]].to(dev).expand(len(xb), -1))
            preds.append(torch.sigmoid(o[..., 0].float())[:, 0].cpu())
        p = torch.cat(preds).flatten().numpy()
        g_ = VCROPS[:, k].flatten().numpy()
        rs.append(np.corrcoef(p, g_)[0, 1] if p.std() > 0 and g_.std() > 0 else np.nan)
    net.train()
    return float(np.nanmean(rs)), rs

def step_fn(model, img, opt, scaler, sched):
    ids = IDS.to(dev).expand(len(img), -1)
    img, ids, masked, act = apply_channel_masking(img, ids, 0.75, 0.5, apply_channel_subset_sampling=True)
    masked, _ = apply_spatial_masking(masked, 0.6, 8)
    with autocast(device_type="cuda", dtype=torch.float16):
        out = model(masked, act, ids)["output"]
    mi, logvar = out.float().unbind(dim=-1)
    loss = beta_nll_loss(img, torch.sigmoid(mi), ClampWithGrad.apply(logvar, -15.0, 15.0), beta=0.5)
    if not torch.isfinite(loss):
        opt.zero_grad(set_to_none=True)
        return float("nan")
    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    scaler.step(opt)
    scaler.update()
    opt.zero_grad(set_to_none=True)
    sched.step()
    return float(loss.item())

try:
    log(f"child started: {len(JTR)} train / {len(JVA)} val images, {len(VCROPS)} val crops, {N_GPU} GPUs")
    # time 30 steps with a provisional schedule, then fix the number of steps to the time budget
    net, model = build()
    opt = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)
    scaler = GradScaler()
    sched = get_scheduler_with_warmup(opt, 100, 10_000, final_lr=1e-5, peak_lr=5e-4, type="cosine")
    for _ in range(5):
        step_fn(model, train_batch(), opt, scaler, sched)
    torch.cuda.synchronize()
    t_start = time.time()
    for _ in range(30):
        step_fn(model, train_batch(), opt, scaler, sched)
    torch.cuda.synchronize()
    sec = (time.time() - t_start) / 30
    t_val = time.time()
    validate(net)
    sec_val = time.time() - t_val
    EVAL_EVERY = cfg["eval_every"]
    TOTAL = int(cfg["train_min"] * 60 / (sec + sec_val / EVAL_EVERY))
    log(f"{sec:.3f} s/step, validation {sec_val:.0f} s -> {TOTAL} steps in {cfg['train_min']:.0f} min")
    net, model = build()
    opt = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)
    scaler = GradScaler()
    sched = get_scheduler_with_warmup(opt, int(0.1 * TOTAL), TOTAL - int(0.1 * TOTAL), final_lr=1e-5, peak_lr=5e-4, type="cosine")
    LOGT, best, losses, n_nan = [], -np.inf, [], 0
    t_train = time.time()
    model.train()
    for step in range(1, TOTAL + 1):
        l = step_fn(model, train_batch(), opt, scaler, sched)
        n_nan += int(not np.isfinite(l))
        losses.append(l)
        if step % EVAL_EVERY == 0 or step == TOTAL:
            vr, per = validate(net)
            LOGT.append(dict(step=step, minutes=(time.time() - t_train) / 60, train_loss=float(np.nanmean(losses)),
                             val_loo_pixel_r=vr, nan_steps=n_nan, lr=sched.get_last_lr()[0],
                             **{f"val_r__{CH[k]}": per[k - 3] for k in range(3, C)}))
            losses = []
            if vr > best:
                best = vr
                torch.save({"model_state_dict": {k: v.detach().cpu().clone() for k, v in net.state_dict().items()},
                            "model_config": net.get_architecture_config(), "channels": CH, "clip": CLIP,
                            "code_source": cfg["code_source"], "best_val_loo_pixel_r": best, "step": step,
                            "total_steps": TOTAL, "train_minutes": cfg["train_min"],
                            "n_jackson_train_images": len(JTR), "n_jackson_val_images": len(JVA)},
                           f"{OUT}/immuvis_jackson_best.pth")
            pd.DataFrame(LOGT).to_csv(f"{OUT}/immuvis_training_log.csv", index=False)
            log(f"step {step}/{TOTAL}: train loss {LOGT[-1]['train_loss']:.4f}, val LOO pixel r {vr:.3f} (best {best:.3f})")
    json.dump({"total_steps": TOTAL, "s_per_step": sec, "best_val_loo_pixel_r": best, "nan_steps": n_nan},
              open(f"{OUT}/child_done.json", "w"))
    log("child done")
except Exception:
    LOGF.write("CHILD EXCEPTION\n" + traceback.format_exc())
    raise
"""
open("/tmp/train_immuvis.py", "w").write(SCRIPT)
CFG = dict(root=root, out=OUT, channels=CH, clip=CLIP, crop=CROP, batch=BATCH, seed=SEED, train_min=TRAIN_MIN,
           eval_every=EVAL_EVERY, code_source=CODE_SOURCE,
           enc_args=dict(ma_layers_blocks=[4], ma_embedding_dims=[16], pm_layers_blocks=[4, 4, 4],
                         pm_embedding_dims=[192, 384, 768], hyperkernel=dict(kernel_size=1, padding=0, stride=1, use_bias=True)),
           dec_args=dict(decoded_embed_dim=512, num_blocks=1,
                         hyperkernel=dict(kernel_size=1, padding=0, stride=1, use_bias=True)))
PROG = f"{OUT}/train_progress.log"
open(PROG, "w").close()
child = subprocess.Popen([sys.executable, "/tmp/train_immuvis.py", json.dumps(CFG)],
                         stdout=open(f"{OUT}/child_stdout.log", "w"), stderr=subprocess.STDOUT)
seen, t_launch = 0, time.time()
while child.poll() is None:
    time.sleep(60)
    lines = open(PROG).read().splitlines()
    for ln in lines[seen:]:
        print(ln, flush=True)
    seen = len(lines)
    try:
        crss = psutil.Process(child.pid).memory_info().rss / 2**30
    except psutil.Error:
        crss = float("nan")
    if int(time.time() - t_launch) % 600 < 60:
        print(f"[monitor {(time.time() - t_launch) / 60:.0f} min] child RSS {crss:.1f} GB; {sysmem()}", flush=True)
lines = open(PROG).read().splitlines()
for ln in lines[seen:]:
    print(ln, flush=True)
rc = child.returncode
print(f"child exit code {rc}" + (f" (killed by signal {signal.Signals(-rc).name})" if rc < 0 else ""), flush=True)
print("child stdout tail:\n" + "".join(open(f"{OUT}/child_stdout.log").readlines()[-30:]), flush=True)
json.dump({"code_source": CODE_SOURCE, "train_minutes": TRAIN_MIN, "child_exit_code": rc,
           "minutes": (time.time() - T0) / 60, "n_jackson_train_images": N_TR, "n_jackson_val_images": N_VA,
           **(json.load(open(f"{OUT}/child_done.json")) if os.path.exists(f"{OUT}/child_done.json") else {})},
          open(f"{OUT}/run_info.json", "w"))
assert rc == 0 and os.path.exists(f"{OUT}/immuvis_jackson_best.pth"), "training child failed"
tick("ImmuVis trained")